# Computing Saliency Attribution for Gender-Contrastive Translations

# Install inseq requirements

In [ ]:
!python --version

In [ ]:
pip install inseq

In [ ]:
pip install sacremoses

In [1]:
%%capture
# Run in Colab to install local packages
%pip install bitsandbytes accelerate transformers inseq

In [ ]:
!python -c "import inseq; print(inseq.get_version())"

In [2]:
import inseq
from inseq.data.aggregator import SubwordAggregator

In [ ]:
pip install "numpy<2.1.0" "pandas==2.2.2"

# Open file

## Translate - How the text could be translated (can be skipped for further analysis)

English sentences into German/Spanish using OPUS-MT model

You need to upload the data Excel file: ``source_data-and-contrastive_translations.xlsx``

In [ ]:
from google.colab import files

# Upload file
uploaded = files.upload()

In [5]:
import pandas as pd

# Read the Excel file into a pandas DataFrame
df = pd.read_excel('/content/source_data-and-contrastive_translations.xlsx')

# Access columns and save contents to a list
EN_sentences = df["EN_source"].tolist()

In [ ]:
print(df)

### Translate into German

In [ ]:
import inseq
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

src = "en"  # source language
trg = "de"  # target language

attribution_model = inseq.load_model(f"Helsinki-NLP/opus-mt-{src}-{trg}", "saliency")

tokenizer = AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-{src}-{trg}")

model = AutoModelForSeq2SeqLM.from_pretrained(f"Helsinki-NLP/opus-mt-{src}-{trg}")

In [ ]:
translations_DE = []
for i in range(60):
  # Tokenize the input sentence
  inputs = tokenizer(EN_sentences[i], return_tensors="pt", padding=True)

  # Generate translation
  translated_tokens = model.generate(**inputs)

  # Decode the generated tokens back to text
  translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

  print(f"SENTENCE {i}")
  translations_DE.append(translated_text)
  print(translated_text)

print(translations_DE)


### Translate into Spanish

In [ ]:
import inseq
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

src = "en"  # source language
trg = "es"  # target language

attribution_model = inseq.load_model(f"Helsinki-NLP/opus-mt-{src}-{trg}", "saliency")

tokenizer = AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-{src}-{trg}")

model = AutoModelForSeq2SeqLM.from_pretrained(f"Helsinki-NLP/opus-mt-{src}-{trg}")

In [ ]:
translations_ES = []
for i in range(60):
  # Tokenize the input sentence
  inputs = tokenizer(EN_sentences[i], return_tensors="pt", padding=True)

  # Generate translation
  translated_tokens = model.generate(**inputs)

  # Decode the generated tokens back to text
  translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

  print(f"SENTENCE {i}")
  translations_ES.append(translated_text)
  print(translated_text)

print(translations_ES)

### Concatenate df and translations and save to Excel

In [ ]:
df['OPUS-MT_DE'] = translations_DE
df['OPUS-MT_ES'] = translations_ES

df_concat = df # Now df_concat refers to the modified df
print(df_concat)

df_concat.to_excel("source_data-and-contrastive_translations.xlsx", index=False)

## Open file already provided on GitHub
``source_data-and-contrastive_translations.xlsx``

In [ ]:
from google.colab import files

# Upload file
uploaded = files.upload()

In [ ]:
import pandas as pd

# Read the Excel file into a pandas DataFrame
df = pd.read_excel('source_data-and-contrastive_translations.xlsx')


# Access columns and save contents to a list
EN_source = df["EN_source"].tolist()
Opus_MT_ES = df["OPUS-MT_ES"].tolist()
Opus_MT_ES_contrastive = df["OPUS-MT_ES_contrastive"].tolist()
Opus_MT_DE = df["OPUS-MT_DE"].tolist()
Opus_MT_DE_contrastive = df["OPUS-MT_DE_contrastive"].tolist()


print(f"EN Source: {EN_source[1]}")
print(f"OPUS-MT ES Translation: {Opus_MT_ES[1]}")
print(f"OPUS-MT ES Contrastive Translation: {Opus_MT_ES_contrastive[1]}")
print(f"OPUS-MT DE Translation: {Opus_MT_DE[1]}")
print(f"OPUS-MT DE Contrastive Translation: {Opus_MT_DE_contrastive[1]}")

EN Source Satistics: print average length of EN source sentences and standard deviation.

In [ ]:
import numpy as np
import pandas as pd

#  Calculate lengths, filter out NaN and keep only valid strings
sentence_lengths = [len(sentence.split()) for sentence in EN_source if isinstance(sentence, str)]

# Calculate statistics
average_length = np.mean(sentence_lengths)
std_length = np.std(sentence_lengths, ddof=1)  # ddof=1 for sample std deviation

# Print results
print("=== SENTENCE LENGTH STATISTICS ===")
print(f"Average length: {average_length:.2f} words")
print(f"Standard deviation: {std_length:.2f} words")
print(f"Min length: {min(sentence_lengths)} words")
print(f"Max length: {max(sentence_lengths)} words")
print(f"Total sentences: {len(sentence_lengths)}")
print(f"Sentences with NaN values skipped: {len(EN_source) - len(sentence_lengths)}")

# Perform the Contrastive Attribution

We use [inseq](https://github.com/inseq-team/inseq) to compute saliency attribution scores for gender contrastive translations.

Load the Attribution Model: We chose Helsinki's OPUS-MT en-de/es (same as used for the translations), and we focus on 'saliency'.

In [ ]:
import inseq

src = "en"  # source language
trg = "es"  # target language ---> Change to "DE" for German

attribution_model = inseq.load_model(f"Helsinki-NLP/opus-mt-{src}-{trg}", "saliency")

### Attribution Visualisation
The cell below visualises inseq's display of saliency attribution for source and target tokens for our given English source and Spanish target sentence. For this project, we focus on the source attribution.

In [22]:
# To visualise
# We access and visualise for Spanish here

# Tests just the first few sentences
for i in range(5):
  en = EN_source[i]
  de_orig = Opus_MT_ES[i]
  de_cont = Opus_MT_ES_contrastive[i]


  # Gets original tokens in format [(0, '</s>'), (1, 'deu_Latn'), (2, '▁Beispi'), (3, 'els'), (4, 'atz'), (5, '▁auf'), (6, '▁Eng'), (7, 'lis'), (8, 'ch'), (9, ','), (10, '▁um'), (11, '▁zu'), (12, '▁zeigen'), (13, ','), (14, '▁wes'), (15, 'halb'), (16, '▁dies'), (17, '▁ein'), (18, '▁Problem'), (19, '▁für'), (20, '▁den'), (21, '▁Program'), (22, 'mi'), (23, 'erer'), (24, '▁ist'), (25, '.'), (26, '</s>')]
  orig_tokens = list(enumerate(attribution_model.encode(de_orig, as_targets=True).input_tokens[0]))

  out = attribution_model.attribute(
      en,
      de_orig ,
      contrast_targets=de_cont,
      attribute_target=True,
      attributed_fn="contrast_prob_diff",
      step_scores=["contrast_prob_diff"],
      contrast_targets_alignments=[(i,i) for i in range(len(orig_tokens))],
  )

  out.show()

Attributing with saliency...:   5%|▍         | 1/21 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/inseq/utils/contrast_utils.py:155: UserWarning: Contrastive inputs do not match original inputs when using a contrastive attributed function.
By default we force the original inputs to be used (i.e. only the contrastive predicted target is different).
This is a requirement for gradient-based attribution method, as contrastive inputs don't participate in gradient computation.
For attribution methods with less stringent requirements, set --contrast_force_inputs to True to use the contrastive inputs for attribution instead.
  warnings.warn(
Attributing with saliency...: 100%|██████████| 21/21 [00:01<00:00, 11.07it/s]


,,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
,,▁Lo,▁que,▁nunca,▁deberías,▁hacer,▁en,▁un,▁avión,:,▁La → ▁El,▁asistente,▁de,▁vuelo,▁revela,▁secretos,▁de,▁la,▁aerolínea,.,</s>
0,▁What,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,▁you,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.032,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,▁should,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,▁never,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,▁do,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,▁on,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,▁a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,▁plane,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,:,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.036,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Attributing with saliency...: 100%|██████████| 34/34 [00:02<00:00, 14.16it/s]


,,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32
,,▁En,▁un,▁video,▁de,▁Instagram,▁publicado,▁el,▁mes,▁pasado,",",▁la → ▁el,▁música → ▁músico,"▁""",A,ll,▁To,o,▁Well,"""",▁puede,▁verse,▁colaborando,▁con,▁el,▁productor,▁Jack,▁Anton,off,▁en,▁el,▁piano,.,</s>
0,▁In,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.008,0.004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,▁an,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.008,0.008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,▁Instagram,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.028,0.019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,▁video,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.011,0.012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,▁posted,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.025,0.014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,▁last,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.011,0.007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,▁month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.012,0.013,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,",",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.01,0.006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,▁the,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.023,0.023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Attributing with saliency...: 100%|██████████| 28/28 [00:01<00:00, 14.88it/s]


,,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26
,,▁El,▁jueves,▁por,▁la,▁noche,",",▁finalmente,",",▁salió,▁a,▁la,▁cancha,▁contra,▁una → ▁un,▁oponente,▁entre,▁los,▁10,▁mejores,▁por,▁segunda,▁vez,▁de,▁su,▁vida,.,</s>
0,▁On,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,▁Thursday,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,▁evening,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,",",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,▁finally,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,",",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,▁she,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.018,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,▁stepped,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,▁out,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Attributing with saliency...: 100%|██████████| 66/66 [00:04<00:00, 13.45it/s]


,,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64
,,▁El → ▁La,▁social,ita,▁que,▁consiguió,▁que,▁una,▁con,desa,▁escribi,era,▁un,▁manual,▁de,▁50,▁páginas,▁sobre,▁cosas,▁como,▁lo,▁llena,▁que,▁tenía,▁que,▁estar,▁una,▁caja,▁de,▁tejidos,▁antes,▁de,▁ser,▁de,se,cha,da,▁-,▁la,▁mitad,▁-,▁nunca,▁no,tó,▁la,▁línea,▁de,▁producción,▁de,▁la,▁fábrica,▁de,▁niños,▁recién,▁nacidos,▁de,▁chicas,▁menores,▁de,▁edad,▁que,▁iban,▁y,▁venían,.,</s>
0,▁The,0.041,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,▁social,0.142,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ite,0.272,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,▁who,0.049,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,▁got,0.023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,▁a,0.014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,▁count,0.046,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,es,0.032,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,s,0.017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Attributing with saliency...: 100%|██████████| 21/21 [00:01<00:00, 15.05it/s]


,,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
,,▁Te,ra,pe,uta,▁de,▁masaje,▁Ken,s,ington,▁encarcelado → ▁encarcela,▁por → da,▁a → ▁por,gre → ▁a,dir → gre,▁sexualmente → dir,▁a → ▁sexualmente,▁clientes → ▁a,. → ▁clientes,</s> → .,</s>
0,▁Ken,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016,0.009,0.024,0.01,0.013,0.006,0.005,0.006,0.006,0.003,0.0
1,s,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.011,0.007,0.006,0.005,0.004,0.003,0.002,0.004,0.003,0.002,0.0
2,ington,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.092,0.013,0.027,0.017,0.013,0.008,0.01,0.009,0.009,0.005,0.0
3,▁massage,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.032,0.048,0.046,0.037,0.025,0.028,0.017,0.028,0.021,0.027,0.0
4,▁therapist,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.028,0.103,0.045,0.043,0.038,0.027,0.025,0.031,0.03,0.036,0.0
5,▁jail,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.014,0.033,0.075,0.031,0.038,0.028,0.014,0.029,0.024,0.011,0.0
6,ed,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005,0.01,0.016,0.008,0.009,0.007,0.003,0.009,0.005,0.003,0.0
7,▁for,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005,0.007,0.042,0.017,0.014,0.018,0.004,0.019,0.01,0.005,0.0
8,▁sexually,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.01,0.016,0.072,0.085,0.055,0.069,0.021,0.122,0.049,0.023,0.0


## Pre-Processing Functions

### 1. Function that prints contrastive probability difference & source attribution for all target words

In [23]:
def print_all_source_attributions_for_all_targets(out):
    """
    Prints all source token attributions (in original source order) for every target token.
    Handles alignment between target tokens and the attribution matrix using attr_pos_start and attr_pos_end.
    """
    out_data = out.sequence_attributions[0]
    source_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.source]
    target_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.target]

    # Handle alignment window (important for correct attribution indexing)
    attr_pos_start = getattr(out_data, "attr_pos_start", 0)
    attr_pos_end = getattr(out_data, "attr_pos_end", len(target_tokens))
    aligned_target_tokens = target_tokens[attr_pos_start:attr_pos_end]  # Only tokens with attributions

    # Prepare attribution matrix (handle possible 3D tensor)
    saliency_heatmap = out_data.source_attributions
    if hasattr(saliency_heatmap, "shape") and len(saliency_heatmap.shape) == 3:
        saliency_heatmap = saliency_heatmap.sum(dim=2)

    contrast_scores = out_data.step_scores.get("contrast_prob_diff", [None] * len(aligned_target_tokens))

    for target_word_index, tgt_token in enumerate(aligned_target_tokens):
        contrast_val = float(contrast_scores[target_word_index]) if contrast_scores[target_word_index] is not None else None
        print(f"Target word (index {target_word_index + attr_pos_start}): '{tgt_token}'")
        if contrast_val is not None:
            print(f"Contrastive prob diff: {contrast_val:.6f}")
        print("All source token attributions (original order):")
        for i, src_token in enumerate(source_tokens):
            score = float(saliency_heatmap[i][target_word_index].item() if hasattr(saliency_heatmap[i][target_word_index], "item") else saliency_heatmap[i][target_word_index])
            print(f"  Source token '{src_token}': attribution score = {score:.6f}")
        print("")

In [24]:
print_all_source_attributions_for_all_targets(out)

Target word (index 1): '▁Te'
Contrastive prob diff: 0.000000
All source token attributions (original order):
  Source token '▁Ken': attribution score = 0.000000
  Source token 's': attribution score = 0.000000
  Source token 'ington': attribution score = 0.000000
  Source token '▁massage': attribution score = 0.000000
  Source token '▁therapist': attribution score = 0.000000
  Source token '▁jail': attribution score = 0.000000
  Source token 'ed': attribution score = 0.000000
  Source token '▁for': attribution score = 0.000000
  Source token '▁sexually': attribution score = 0.000000
  Source token '▁assault': attribution score = 0.000000
  Source token 'ing': attribution score = 0.000000
  Source token '▁clients': attribution score = 0.000000
  Source token '.': attribution score = 0.000000
  Source token '</s>': attribution score = 0.000000

Target word (index 2): 'ra'
Contrastive prob diff: 0.000000
All source token attributions (original order):
  Source token '▁Ken': attribution sc

### 2. A function that prints all source attributions for the first target word with a translation "switch"

(e.g. "encarcelado" → "encarcela")

In [25]:
def print_all_source_attributions_for_first_arrow_word(out):
    """
    Prints all source token attributions (in original source order) for the first target token containing '→'.
    Handles alignment between target tokens and the attribution matrix using attr_pos_start and attr_pos_end.
    """
    out_data = out.sequence_attributions[0]
    source_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.source]
    target_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.target]

    # Handle alignment window (important for correct attribution indexing)
    attr_pos_start = getattr(out_data, "attr_pos_start", 0)
    attr_pos_end = getattr(out_data, "attr_pos_end", len(target_tokens))
    aligned_target_tokens = target_tokens[attr_pos_start:attr_pos_end]  # Only tokens with attributions

    # Prepare attribution matrix (handle possible 3D tensor)
    saliency_heatmap = out_data.source_attributions
    if hasattr(saliency_heatmap, "shape") and len(saliency_heatmap.shape) == 3:
        saliency_heatmap = saliency_heatmap.sum(dim=2)

    contrast_scores = out_data.step_scores["contrast_prob_diff"]

    # Find first target token containing "→" in the aligned window
    target_word_index = next((i for i, token in enumerate(aligned_target_tokens) if "→" in token), None)
    if target_word_index is None:
        print("No target token containing '→' found.")
        return

    tgt_token = aligned_target_tokens[target_word_index]
    contrast_val = float(contrast_scores[target_word_index])

    print(f"First target word containing '→' (index {target_word_index + attr_pos_start}): '{tgt_token}'")
    print(f"Contrastive prob diff: {contrast_val:.6f}")
    print("\nAll source token attributions (original order):")

    # Attribution scores for this target token (column in matrix), in original source token order
    for i, src_token in enumerate(source_tokens):
        score = float(saliency_heatmap[i][target_word_index].item() if hasattr(saliency_heatmap[i][target_word_index], "item") else saliency_heatmap[i][target_word_index])
        print(f"  Source token '{src_token}': attribution score = {score:.6f}")
    print("")


In [26]:
print_all_source_attributions_for_first_arrow_word(out)

First target word containing '→' (index 10): '▁encarcelado → ▁encarcela'
Contrastive prob diff: 0.000001

All source token attributions (original order):
  Source token '▁Ken': attribution score = 0.000065
  Source token 's': attribution score = 0.000046
  Source token 'ington': attribution score = 0.000398
  Source token '▁massage': attribution score = 0.000141
  Source token '▁therapist': attribution score = 0.000124
  Source token '▁jail': attribution score = 0.000062
  Source token 'ed': attribution score = 0.000021
  Source token '▁for': attribution score = 0.000021
  Source token '▁sexually': attribution score = 0.000044
  Source token '▁assault': attribution score = 0.000030
  Source token 'ing': attribution score = 0.000007
  Source token '▁clients': attribution score = 0.000030
  Source token '.': attribution score = 0.000022
  Source token '</s>': attribution score = 0.000074



### 3. Same function, now adapted to add source attributions per target word up to 1 (so they can be interpreted as probabilities and can be compared across examples) --> normalisation


In [27]:
def print_normalized_source_attributions_for_first_arrow_word(out):
    """
    Prints the source token attributions for the first target token containing '→',
    after normalizing source attributions for that target token so that they sum to 1.
    Also prints the index of each source token.
    Handles alignment between target tokens and the attribution matrix using attr_pos_start and attr_pos_end.
    """
    out_data = out.sequence_attributions[0]
    source_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.source]
    target_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.target]

    # Handle alignment window (important for correct attribution indexing)
    attr_pos_start = getattr(out_data, "attr_pos_start", 0)
    attr_pos_end = getattr(out_data, "attr_pos_end", len(target_tokens))
    aligned_target_tokens = target_tokens[attr_pos_start:attr_pos_end]  # Only tokens with attributions

    # Prepare attribution matrix (handle possible 3D tensor)
    saliency_heatmap = out_data.source_attributions
    if hasattr(saliency_heatmap, "shape") and len(saliency_heatmap.shape) == 3:
        saliency_heatmap = saliency_heatmap.sum(dim=2)

    contrast_scores = out_data.step_scores["contrast_prob_diff"]

    # Find first target token containing "→" in the aligned window
    target_word_index = next((i for i, token in enumerate(aligned_target_tokens) if "→" in token), None)
    if target_word_index is None:
        print("No target token containing '→' found.")
        return

    tgt_token = aligned_target_tokens[target_word_index]
    contrast_val = float(contrast_scores[target_word_index])

    # Collect and normalize attribution scores for this target token (column in matrix)
    raw_scores = [
        float(saliency_heatmap[i][target_word_index].item() if hasattr(saliency_heatmap[i][target_word_index], "item") else saliency_heatmap[i][target_word_index])
        for i in range(len(source_tokens))
    ]
    score_sum = sum(raw_scores)
    normalized_scores = [s / score_sum if score_sum != 0 else 0.0 for s in raw_scores]

    print(f"First target word containing '→' (index {target_word_index + attr_pos_start}): '{tgt_token}'")
    print(f"Contrastive prob diff: {contrast_val:.6f}")

    # Sort by normalized score, descending, but print index of original
    source_attributions = list(enumerate(zip(source_tokens, normalized_scores)))
    source_attributions.sort(key=lambda x: x[1][1], reverse=True)
    for idx, (src_token, score) in source_attributions:
        print(f"  Source token idx {idx:2d} '{src_token}': normalized attribution score = {score:.6f}")
    print("")

    # Optionally, print all normalized attributions in original order for inspection
    print("All source tokens and their normalized attributions (original order):")
    for idx, (src_token, score) in enumerate(zip(source_tokens, normalized_scores)):
        print(f"  Source token idx {idx:2d} '{src_token}': normalized attribution score = {score:.6f}")
    print("")

In [28]:
print_normalized_source_attributions_for_first_arrow_word(out)

First target word containing '→' (index 10): '▁encarcelado → ▁encarcela'
Contrastive prob diff: 0.000001
  Source token idx  2 'ington': normalized attribution score = 0.366875
  Source token idx  3 '▁massage': normalized attribution score = 0.129736
  Source token idx  4 '▁therapist': normalized attribution score = 0.114071
  Source token idx 13 '</s>': normalized attribution score = 0.067793
  Source token idx  0 '▁Ken': normalized attribution score = 0.060252
  Source token idx  5 '▁jail': normalized attribution score = 0.057250
  Source token idx  1 's': normalized attribution score = 0.042765
  Source token idx  8 '▁sexually': normalized attribution score = 0.040379
  Source token idx 11 '▁clients': normalized attribution score = 0.027248
  Source token idx  9 '▁assault': normalized attribution score = 0.027228
  Source token idx 12 '.': normalized attribution score = 0.020617
  Source token idx  6 'ed': normalized attribution score = 0.019530
  Source token idx  7 '▁for': normali

### 4. Same function, which additionally merges subwords and source contributions, and removes target referents

Defines function including removing target words (iteratively, first target word removed from first sentence, second target word removed from second sentence etc.)

In [29]:
def print_normalized_merged_source_attributions_for_first_arrow_word(out, sentence_index, target_words):
    """
    Prints the merged source word attributions for the first target token containing '→',
    after normalizing source attributions for that target token so that they sum to 1.
    Subword tokens are merged into whole words before ranking and printing.
    Also prints the indices of the constituent source tokens.
    Removes the specific target word for this sentence based on sentence_index.
    """
    out_data = out.sequence_attributions[0]
    # Get source tokens and optionally their ids and original index
    source_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.source]
    target_tokens = [t.token if hasattr(t, 'token') else str(t) for t in out_data.target]

    print(f"\n--- SENTENCE {sentence_index + 1} ---")
    print(source_tokens)
    print(target_tokens)

    # Remove end of line character from source tokens
    if "</s>" in source_tokens:
        source_tokens.remove("</s>")

    # Handle alignment window (important for correct attribution indexing)
    attr_pos_start = getattr(out_data, "attr_pos_start", 0)
    attr_pos_end = getattr(out_data, "attr_pos_end", len(target_tokens))
    aligned_target_tokens = target_tokens[attr_pos_start:attr_pos_end]  # Only tokens with attributions

    # Prepare attribution matrix (handle possible 3D tensor)
    saliency_heatmap = out_data.source_attributions
    if hasattr(saliency_heatmap, "shape") and len(saliency_heatmap.shape) == 3:
        saliency_heatmap = saliency_heatmap.sum(dim=2)

    contrast_scores = out_data.step_scores["contrast_prob_diff"]

    # Find first target token containing "→" in the aligned window
    target_word_index = next((i for i, token in enumerate(aligned_target_tokens) if "→" in token), None)
    if target_word_index is None:
        print("No target token containing '→' found.")
        return

    tgt_token = aligned_target_tokens[target_word_index]
    contrast_val = float(contrast_scores[target_word_index])

    # Collect and normalize attribution scores for this target token (column in matrix)
    raw_scores = [
        float(saliency_heatmap[i][target_word_index].item() if hasattr(saliency_heatmap[i][target_word_index], "item") else saliency_heatmap[i][target_word_index])
        for i in range(len(source_tokens))
    ]
    score_sum = sum(raw_scores)
    normalized_scores = [s / score_sum if score_sum != 0 else 0.0 for s in raw_scores]

    # Merge subwords into words and sum their attribution scores
    merged_words = []
    current_word = ""
    current_score = 0.0
    current_indices = []

    for idx, (token, score) in enumerate(zip(source_tokens, normalized_scores)):
        if token.startswith("▁") or token in [".", "</s>"]:  # Start of a new word or special
            if current_word:
                merged_words.append((current_word, current_score, current_indices))
            # Start new word
            current_word = token.lstrip("▁")
            current_score = score
            current_indices = [idx]
        else:
            current_word += token
            current_score += score
            current_indices.append(idx)
    # Add last word
    if current_word:
        merged_words.append((current_word, current_score, current_indices))


    # Get the specific target word for this sentence
    current_target_word = target_words.get(sentence_index, "").lower()
    print(f"Target word for sentence {sentence_index + 1}: '{current_target_word}'")

    # Create a new list excluding the specific target word for this sentence
    filtered_words = []

    # Extract individual words from compound target words
    all_target_words = set()
    if current_target_word:
        all_target_words.add(current_target_word)  # Add the full compound
        # Add individual words from compounds
        for word_part in current_target_word.split():
            all_target_words.add(word_part)

    print(f"All target words to remove (including compound parts): {all_target_words}")

    for word, score, indices in merged_words:
        word_stripped = word.strip(".,:\"'—.-?![]()")
        word_lower = word_stripped.lower()

        # Check if the word itself is in target_words (including compound parts)
        if word_lower in all_target_words:
            print(f"Removing '{word_stripped}' - found in target_words")
            continue

        # Check if any individual word from a multi-word phrase is in target_words
        word_parts = word_lower.split()
        if any(part in all_target_words for part in word_parts):
            print(f"Removing '{word_stripped}' - contains part in target_words")
            continue

        # If word doesn't match any target words, keep it
        filtered_words.append((word_stripped, score, indices))

    merged_words = filtered_words

    # Sort merged words by total attribution score, descending
    merged_words.sort(key=lambda x: x[1], reverse=True)

    print(f"\nFirst target word containing '→' (index {target_word_index + attr_pos_start}): '{tgt_token}'")
    print(f"Contrastive prob diff: {contrast_val:.6f}")

    for word, score, indices in merged_words:
        print(f"  Source word '{word}' (source token idx {indices}): normalized attribution score = {score:.6f}")
    print("")

Run function to remove target words

In [31]:
# Your target words dictionary
target_words = {0: 'flight attendant', 1: 'musician', 2: 'opponent', 3: 'socialite', 4: 'therapist', 5: 'coordinator', 6: 'lover', 7: 'mechanic', 8: 'dancer', 9: 'therapist', 10: 'visitor', 11: 'colleague', 12: 'companion', 13: 'author',
                14: 'clerk', 15: 'student', 16: 'accountant', 17: 'designer', 18: 'baker', 19: 'lover', 20: 'writer', 21: 'winner', 22: 'consumer', 23: 'poet', 24: 'author', 25: 'writer', 26: 'designer', 27: 'bookkeeper', 28: 'clerk',
                29: 'author', 30: 'counselor', 31: 'dancer', 32: 'friend', 33: 'guard', 34: 'officer', 35: 'winner', 36: 'user', 37: 'supporter', 38: 'judge', 39: 'fighter', 40: 'dealer', 41: 'soldier', 42: 'officer', 43: 'player',
                44: 'manager', 45: 'contractor', 46: 'captain', 47: 'farmer', 48: 'maestro', 49: 'construction worker', 50: 'boss', 51: 'driver', 52: 'idiot', 53: 'cook', 54: 'filmmaker', 55: 'admirer', 56: 'follower', 57: 'salesperson',
                58: 'buddy', 59: 'winner'}

# Updated main loop
for i in range(5):  # Change to range(59) to iterate through all 60 sentences
    en = EN_source[i]
    de_orig = Opus_MT_ES[i]
    de_cont = Opus_MT_ES_contrastive[i]

    # Gets original tokens
    orig_tokens = list(enumerate(attribution_model.encode(de_orig, as_targets=True).input_tokens[0]))

    out = attribution_model.attribute(
        en,
        de_orig,
        contrast_targets=de_cont,
        attribute_target=True,
        attributed_fn="contrast_prob_diff",
        step_scores=["contrast_prob_diff"],
        contrast_targets_alignments=[(i,i) for i in range(len(orig_tokens))],
    )

    # Pass the sentence index and target_words dictionary to the function
    print_normalized_merged_source_attributions_for_first_arrow_word(out, i, target_words)

Attributing with saliency...: 100%|██████████| 21/21 [00:01<00:00, 11.60it/s]



--- SENTENCE 1 ---
['▁What', '▁you', '▁should', '▁never', '▁do', '▁on', '▁a', '▁plane', ':', '▁Flight', '▁attendant', '▁reveals', '▁airline', '▁secrets', '.', '</s>']
['<pad>', '▁Lo', '▁que', '▁nunca', '▁deberías', '▁hacer', '▁en', '▁un', '▁avión', ':', '▁La → ▁El', '▁asistente', '▁de', '▁vuelo', '▁revela', '▁secretos', '▁de', '▁la', '▁aerolínea', '.', '</s>']
Target word for sentence 1: 'flight attendant'
All target words to remove (including compound parts): {'flight', 'flight attendant', 'attendant'}
Removing 'Flight' - found in target_words
Removing 'attendant' - found in target_words

First target word containing '→' (index 10): '▁La → ▁El'
Contrastive prob diff: 0.356609
  Source word 'reveals' (source token idx [11]): normalized attribution score = 0.079459
  Source word 'plane' (source token idx [7, 8]): normalized attribution score = 0.077359
  Source word 'airline' (source token idx [12]): normalized attribution score = 0.072081
  Source word 'secrets' (source token idx [13]

Attributing with saliency...: 100%|██████████| 34/34 [00:02<00:00, 14.65it/s]



--- SENTENCE 2 ---
['▁In', '▁an', '▁Instagram', '▁video', '▁posted', '▁last', '▁month', ',', '▁the', '▁"', 'All', '▁Too', '▁Well', '"', '▁musician', '▁can', '▁be', '▁seen', '▁collaborating', '▁with', '▁producer', '▁Jack', '▁Anton', 'off', '▁on', '▁the', '▁piano', '.', '</s>']
['<pad>', '▁En', '▁un', '▁video', '▁de', '▁Instagram', '▁publicado', '▁el', '▁mes', '▁pasado', ',', '▁la → ▁el', '▁música → ▁músico', '▁"', 'A', 'll', '▁To', 'o', '▁Well', '"', '▁puede', '▁verse', '▁colaborando', '▁con', '▁el', '▁productor', '▁Jack', '▁Anton', 'off', '▁en', '▁el', '▁piano', '.', '</s>']
Target word for sentence 2: 'musician'
All target words to remove (including compound parts): {'musician'}
Removing 'musician' - found in target_words

First target word containing '→' (index 11): '▁la → ▁el'
Contrastive prob diff: 0.682407
  Source word 'collaborating' (source token idx [18]): normalized attribution score = 0.117007
  Source word 'seen' (source token idx [17]): normalized attribution score = 0.09

Attributing with saliency...: 100%|██████████| 28/28 [00:01<00:00, 14.61it/s]



--- SENTENCE 3 ---
['▁On', '▁Thursday', '▁evening', ',', '▁finally', ',', '▁she', '▁stepped', '▁out', '▁onto', '▁the', '▁court', '▁against', '▁a', '▁top', '▁10', '▁opponent', '▁for', '▁just', '▁the', '▁second', '▁time', '▁of', '▁her', '▁life', '.', '</s>']
['<pad>', '▁El', '▁jueves', '▁por', '▁la', '▁noche', ',', '▁finalmente', ',', '▁salió', '▁a', '▁la', '▁cancha', '▁contra', '▁una → ▁un', '▁oponente', '▁entre', '▁los', '▁10', '▁mejores', '▁por', '▁segunda', '▁vez', '▁de', '▁su', '▁vida', '.', '</s>']
Target word for sentence 3: 'opponent'
All target words to remove (including compound parts): {'opponent'}
Removing 'opponent' - found in target_words

First target word containing '→' (index 14): '▁una → ▁un'
Contrastive prob diff: 0.551618
  Source word 'top' (source token idx [14]): normalized attribution score = 0.118847
  Source word 'against' (source token idx [12]): normalized attribution score = 0.061687
  Source word '10' (source token idx [15]): normalized attribution score = 

Attributing with saliency...: 100%|██████████| 66/66 [00:04<00:00, 13.75it/s]



--- SENTENCE 4 ---
['▁The', '▁social', 'ite', '▁who', '▁got', '▁a', '▁count', 'es', 's', '▁to', '▁write', '▁a', '▁50', '-', 'page', '▁manual', '▁on', '▁such', '▁things', '▁as', '▁how', '▁full', '▁a', '▁box', '▁of', '▁tissues', '▁had', '▁to', '▁be', '▁before', '▁it', '▁was', '▁thrown', '▁away', '▁-', '▁half', '▁-', '▁never', '▁noticed', '▁the', '▁fresh', '▁child', '▁factory', '▁production', '▁line', '▁of', '▁under', 'age', '▁girls', '▁coming', '▁and', '▁going', '.', '</s>']
['<pad>', '▁El → ▁La', '▁social', 'ita', '▁que', '▁consiguió', '▁que', '▁una', '▁con', 'desa', '▁escribi', 'era', '▁un', '▁manual', '▁de', '▁50', '▁páginas', '▁sobre', '▁cosas', '▁como', '▁lo', '▁llena', '▁que', '▁tenía', '▁que', '▁estar', '▁una', '▁caja', '▁de', '▁tejidos', '▁antes', '▁de', '▁ser', '▁de', 'se', 'cha', 'da', '▁-', '▁la', '▁mitad', '▁-', '▁nunca', '▁no', 'tó', '▁la', '▁línea', '▁de', '▁producción', '▁de', '▁la', '▁fábrica', '▁de', '▁niños', '▁recién', '▁nacidos', '▁de', '▁chicas', '▁menores', '▁de', 

Attributing with saliency...: 100%|██████████| 21/21 [00:01<00:00, 15.80it/s]


--- SENTENCE 5 ---
['▁Ken', 's', 'ington', '▁massage', '▁therapist', '▁jail', 'ed', '▁for', '▁sexually', '▁assault', 'ing', '▁clients', '.', '</s>']
['<pad>', '▁Te', 'ra', 'pe', 'uta', '▁de', '▁masaje', '▁Ken', 's', 'ington', '▁encarcelado → ▁encarcela', '▁por → da', '▁a → ▁por', 'gre → ▁a', 'dir → gre', '▁sexualmente → dir', '▁a → ▁sexualmente', '▁clientes → ▁a', '. → ▁clientes', '</s> → .', '</s>']
Target word for sentence 5: 'therapist'
All target words to remove (including compound parts): {'therapist'}
Removing 'therapist' - found in target_words

First target word containing '→' (index 10): '▁encarcelado → ▁encarcela'
Contrastive prob diff: 0.000001
  Source word 'Kensington' (source token idx [0, 1, 2]): normalized attribution score = 0.504064
  Source word 'massage' (source token idx [3]): normalized attribution score = 0.139171
  Source word 'jailed' (source token idx [5, 6]): normalized attribution score = 0.082363
  Source word 'sexually' (source token idx [8]): normalized 